# 03 — Transfer Learning: MobileNetV2 & EfficientNetB0

Brain Tumor MRI Classification · Internship Project

## Setup

In [ ]:
import os, sys, random
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sys.path.insert(0, os.path.abspath('..'))
from src.config import (MOBILENET_PATH, EFFICIENTNET_PATH, CLASS_NAMES,
                        DISPLAY_NAMES, EPOCHS_FROZEN, EPOCHS_FINETUNE, TRAIN_DIR)
from src.preprocessing import build_generators
from src.models        import (build_mobilenetv2, fine_tune_mobilenetv2,
                               build_efficientnetb0, fine_tune_efficientnetb0,
                               get_callbacks)
from src.evaluate      import (evaluate_model, plot_confusion_matrix,
                               plot_training_history, save_classification_report,
                               find_misclassified)
from src.data_utils    import compute_class_weights

cw = compute_class_weights(TRAIN_DIR, CLASS_NAMES)
print("Class weights:", cw)

## 1. MobileNetV2

### 1a. Build & Phase-1 Training (Frozen Base)

In [ ]:
train_gen, valid_gen, test_gen = build_generators('mobilenetv2')
mv2 = build_mobilenetv2()
mv2.summary()

In [ ]:
print("\n[MobileNetV2] Phase 1 — training head only …")
h1 = mv2.fit(
    train_gen,
    epochs=EPOCHS_FROZEN,
    validation_data=valid_gen,
    callbacks=get_callbacks(MOBILENET_PATH),
    class_weight=cw,
    verbose=1,
)

### 1b. Fine-Tuning (Unfreeze Top Layers)

In [ ]:
mv2 = tf.keras.models.load_model(MOBILENET_PATH)   # best from phase 1
mv2 = fine_tune_mobilenetv2(mv2)
print(f"Trainable params after unfreezing: {sum(v.numpy().size for v in mv2.trainable_weights):,}")

print("\n[MobileNetV2] Phase 2 — fine-tuning …")
h2 = mv2.fit(
    train_gen,
    epochs=EPOCHS_FINETUNE,
    validation_data=valid_gen,
    callbacks=get_callbacks(MOBILENET_PATH),
    class_weight=cw,
    verbose=1,
)

### 1c. Evaluation

In [ ]:
mv2 = tf.keras.models.load_model(MOBILENET_PATH)
merged = {k: h1.history[k] + h2.history[k] for k in h1.history}
plot_training_history(merged, 'MobileNetV2')

result_mv2 = evaluate_model(mv2, test_gen, 'MobileNetV2')
plot_confusion_matrix(result_mv2['y_true'], result_mv2['y_pred'], 'MobileNetV2')
save_classification_report(result_mv2['report'], 'MobileNetV2')
find_misclassified(mv2, test_gen, 'MobileNetV2')

## 2. EfficientNetB0

### 2a. Build & Phase-1 Training (Frozen Base)

In [ ]:
train_gen_e, valid_gen_e, test_gen_e = build_generators('efficientnetb0')
eff = build_efficientnetb0()
eff.summary()

In [ ]:
print("\n[EfficientNetB0] Phase 1 — training head only …")
h1e = eff.fit(
    train_gen_e,
    epochs=EPOCHS_FROZEN,
    validation_data=valid_gen_e,
    callbacks=get_callbacks(EFFICIENTNET_PATH),
    class_weight=cw,
    verbose=1,
)

### 2b. Fine-Tuning

In [ ]:
eff = tf.keras.models.load_model(EFFICIENTNET_PATH)
eff = fine_tune_efficientnetb0(eff)

print("\n[EfficientNetB0] Phase 2 — fine-tuning …")
h2e = eff.fit(
    train_gen_e,
    epochs=EPOCHS_FINETUNE,
    validation_data=valid_gen_e,
    callbacks=get_callbacks(EFFICIENTNET_PATH),
    class_weight=cw,
    verbose=1,
)

### 2c. Evaluation

In [ ]:
eff = tf.keras.models.load_model(EFFICIENTNET_PATH)
merged_e = {k: h1e.history[k] + h2e.history[k] for k in h1e.history}
plot_training_history(merged_e, 'EfficientNetB0')

result_eff = evaluate_model(eff, test_gen_e, 'EfficientNetB0')
plot_confusion_matrix(result_eff['y_true'], result_eff['y_pred'], 'EfficientNetB0')
save_classification_report(result_eff['report'], 'EfficientNetB0')
find_misclassified(eff, test_gen_e, 'EfficientNetB0')

## Summary

In [ ]:
print("MobileNetV2 Test Results:")
print(f"  Accuracy  : {result_mv2['accuracy']:.4f}")
print(f"  F1-score  : {result_mv2['f1_score']:.4f}")
print()
print("EfficientNetB0 Test Results:")
print(f"  Accuracy  : {result_eff['accuracy']:.4f}")
print(f"  F1-score  : {result_eff['f1_score']:.4f}")